# Qwen2.5-1.5B + LoRA — Host model trên Kaggle (vLLM + ngrok)

Notebook **chỉ để serve** adapter đã train (không train lại). Phơi 1 endpoint OpenAI-compatible qua ngrok để gọi từ máy ngoài (vd benchmark trên Mac).

**Settings (panel bên phải):**
- Accelerator: `GPU T4 x1` (hoặc P100) — 1.5B chỉ cần 1 GPU.
- Internet: **On** (bắt buộc — `pip install`, tải base model, ngrok).
- Add-ons → Secrets: thêm `NGROK_AUTH_TOKEN` (lấy tại https://dashboard.ngrok.com). Nếu load adapter từ HF private repo thì thêm cả `HF_TOKEN`.

**Cách dùng:** chỉnh cell *Cấu hình* bên dưới → Run All → copy `vLLM URL` ở cell cuối. **Giữ kernel chạy**, đóng notebook là URL chết.

## 1. Cấu hình

Chọn nguồn adapter:
- `"dataset"` → adapter nằm trong một Kaggle Dataset đã Add vào notebook (Add Input → chọn Output của notebook train, hoặc dataset bạn tự upload). Sửa `DATASET_DIR` trỏ tới thư mục chứa `adapter_config.json`.
- `"hub"` → tải adapter từ HuggingFace Hub (`HUB_MODEL_ID`). Cần `PUSH_TO_HUB=True` khi train; private repo cần secret `HF_TOKEN`.

In [ ]:
# ===== Cấu hình =====
BASE_MODEL     = "Qwen/Qwen2.5-1.5B-Instruct"   # base model (phải khớp lúc train)
LORA_ID        = "vi-rewriter"                   # tên lora module phơi qua API
MAX_LORA_RANK  = 8                                # = lora_rank lúc train
MAX_MODEL_LEN  = 2048
VLLM_PORT      = 8000

ADAPTER_SOURCE = "dataset"                        # "dataset" | "hub"
# -- khi ADAPTER_SOURCE == "dataset":
DATASET_DIR    = "/kaggle/input/qwen-dialogue-rewriter-lora"  # sửa cho đúng path mount
# -- khi ADAPTER_SOURCE == "hub":
HUB_MODEL_ID   = "ThaoDuongDoingStuff/qwen-vi-rewriter-lora"
print(f"source={ADAPTER_SOURCE} | base={BASE_MODEL} | lora_id={LORA_ID}")

## 2. Cài dependencies

vLLM 0.10.2 + transformers 4.55.4 = combo CUDA-12 ổn định trên driver Kaggle.

In [ ]:
!pip install -q "vllm==0.10.2" "transformers==4.55.4" pyngrok huggingface_hub

## 3. Định vị adapter

Resolve `adapter_dir` (chứa `adapter_config.json`) theo nguồn đã chọn rồi kiểm tra tồn tại.

In [ ]:
import os

try:
    from kaggle_secrets import UserSecretsClient
    _secrets = UserSecretsClient()
except Exception:
    _secrets = None

def _get_secret(name):
    if _secrets is None:
        return None
    try:
        return _secrets.get_secret(name)
    except Exception:
        return None


def _find_adapter(root):
    """Tìm thư mục chứa adapter_config.json (đề phòng dataset lồng 1 lớp)."""
    if os.path.isfile(os.path.join(root, "adapter_config.json")):
        return root
    for dirpath, _dirs, files in os.walk(root):
        if "adapter_config.json" in files:
            return dirpath
    return None


if ADAPTER_SOURCE == "hub":
    from huggingface_hub import snapshot_download
    hf_token = _get_secret("HF_TOKEN")
    adapter_dir = snapshot_download(HUB_MODEL_ID, token=hf_token)
elif ADAPTER_SOURCE == "dataset":
    assert os.path.isdir(DATASET_DIR), (
        f"Không thấy {DATASET_DIR}. Add Input (Output notebook train / dataset adapter) "
        f"rồi sửa DATASET_DIR cho đúng path mount ở /kaggle/input/..."
    )
    adapter_dir = _find_adapter(DATASET_DIR)
else:
    raise ValueError(f"ADAPTER_SOURCE không hợp lệ: {ADAPTER_SOURCE!r}")

assert adapter_dir and os.path.isfile(os.path.join(adapter_dir, "adapter_config.json")), (
    f"Không thấy adapter_config.json (source={ADAPTER_SOURCE}, dir={adapter_dir})."
)
print("adapter_dir:", adapter_dir)
print(os.listdir(adapter_dir))

## 4. Serve LoRA qua vLLM + ngrok

Khởi động vLLM (`--enable-lora`), stream log đến khi server sẵn sàng, rồi mở tunnel ngrok. T4/P100 (cc 7.5/6.0) không hỗ trợ bfloat16 → tự chọn `half`.

In [ ]:
import subprocess, threading, time
import torch
from pyngrok import ngrok

major_cc = torch.cuda.get_device_capability(0)[0] if torch.cuda.is_available() else 0
dtype_flag = "bfloat16" if major_cc >= 8 else "half"
print(f"GPU: {torch.cuda.get_device_name(0)} (cc={major_cc}.x) -> --dtype {dtype_flag}")

# dọn lần chạy cũ (nếu Run lại cell)
subprocess.run(["pkill", "-9", "-f", "vllm serve"], check=False)
try: ngrok.kill()
except Exception: pass
time.sleep(2)

ngrok_token = _get_secret("NGROK_AUTH_TOKEN")
assert ngrok_token, "Thiếu secret NGROK_AUTH_TOKEN (Add-ons → Secrets)."
ngrok.set_auth_token(ngrok_token)

env = dict(os.environ, CUDA_VISIBLE_DEVICES="0")  # 1.5B chỉ cần 1 GPU
vllm_proc = subprocess.Popen(
    ["vllm", "serve", BASE_MODEL,
     "--enable-lora",
     "--lora-modules", f"{LORA_ID}={adapter_dir}",
     "--max-lora-rank", str(MAX_LORA_RANK),
     "--dtype", dtype_flag,
     "--max-model-len", str(MAX_MODEL_LEN),
     "--port", str(VLLM_PORT)],
    stdout=subprocess.PIPE, stderr=subprocess.STDOUT, text=True, bufsize=1, env=env,
)

ready = False
for line in vllm_proc.stdout:           # stream log đến khi sẵn sàng
    print(line, end="")
    if "Application startup complete" in line or "Uvicorn running" in line:
        ready = True; break
    if vllm_proc.poll() is not None: break
if not ready:
    raise RuntimeError(f"vLLM thoát sớm (rc={vllm_proc.wait()}) — xem log phía trên")

url = ngrok.connect(VLLM_PORT).public_url
print("\n================ READY ================")
print("vLLM URL :", url)
print("  base id :", BASE_MODEL)
print("  lora id :", LORA_ID)
print("\n>> Test nhanh từ máy ngoài:")
print(f"   curl {url}/v1/models")
print("\n>> Chạy eval trên Mac:")
print(f"   python -m src.eval.eval_bench --vllm-url {url} \\")
print(f"     --models {LORA_ID} {BASE_MODEL} \\")
print( "     --judge-model mimo-v2.5-pro \\")
print( "     --judge-base-url https://token-plan-sgp.xiaomimimo.com/v1")

# giữ kernel sống + drain log nền để pipe không nghẽn
threading.Thread(target=lambda: [print(l, end="") for l in vllm_proc.stdout], daemon=True).start()

## 5. (Tuỳ chọn) Smoke test ngay trong notebook

Gọi thử endpoint local để chắc adapter trả lời được trước khi dùng URL ngrok.

In [ ]:
import json, urllib.request

SYSTEM_PROMPT = (
    "Bạn là model rewrite hội thoại. Nhiệm vụ của bạn là biến câu nói cuối của user "
    "thành một yêu cầu độc lập, rõ ràng, giữ nguyên ý định, không thêm thông tin "
    "không chắc chắn. Chỉ trả về câu rewrite."
)
payload = {
    "model": LORA_ID,
    "messages": [
        {"role": "system", "content": SYSTEM_PROMPT},
        {"role": "user", "content": "mở điều hoà\nbạn muốn đặt bao nhiêu độ?\n27 độ"},
    ],
    "temperature": 0.1,
    "max_tokens": 96,
}
req = urllib.request.Request(
    f"http://localhost:{VLLM_PORT}/v1/chat/completions",
    data=json.dumps(payload).encode(), headers={"Content-Type": "application/json"},
)
resp = json.loads(urllib.request.urlopen(req).read())
print(resp["choices"][0]["message"]["content"])